In [3]:
#!/usr/bin/env python
import argparse
import sys
from pathlib import Path
import numpy as np
import pandas as pd

HIGHER_IS_BETTER = {"edge_density", "mean_internal_deg", "mean_rwr_score"}
# everything else (conductance, size) is lower-is-better, as in step10


array_dir="06_disease_module/perm_array"
out_dir="06_disease_module/permutation_test_merged"
script_dir="."

array_dir, out_dir = Path(array_dir), Path(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

null_dfs, jac_dfs, true_metrics = [], [], None
for tdir in sorted(array_dir.glob("task_*/permutation_test")):
    nf = tdir / "null_module_metrics.csv"
    jf = tdir / "jaccard_scores.csv"
    rf = tdir / "permutation_test_results.csv"
    if not (nf.exists() and jf.exists() and rf.exists()):
        print(f"  skipping {tdir} (incomplete)")
        continue
    n_df = pd.read_csv(nf)
    null_dfs.append(n_df)
    jac_dfs.append(pd.read_csv(jf))
    if true_metrics is None:  # identical across tasks (deterministic true module)
        res = pd.read_csv(rf)
        true_metrics = dict(zip(res["metric"], res["true_value"]))
    print(f"  {tdir.parent.name}: {len(n_df)} perms")

if not null_dfs:
    print("No completed tasks found.")

null_df = pd.concat(null_dfs, ignore_index=True)
jac = pd.concat(jac_dfs, ignore_index=True)["jaccard"].values
n = len(null_df)
print(f"\nMerged {len(null_dfs)} tasks -> {n} permutations")

pvalues, zscores, rows = {}, {}, []
for m, tv in true_metrics.items():
    v = null_df[m].values
    mu, sd = v.mean(), v.std()
    zscores[m] = (tv - mu) / sd if sd > 0 else np.nan
    pvalues[m] = np.mean(v >= tv) if m in HIGHER_IS_BETTER else np.mean(v <= tv)
    rows.append(dict(metric=m, true_value=tv, null_mean=mu, null_std=sd,
                     z_score=zscores[m], p_value=pvalues[m]))

results = pd.DataFrame(rows)
results.to_csv(out_dir / "permutation_test_results.csv", index=False)
null_df.to_csv(out_dir / "null_module_metrics.csv", index=False)
pd.DataFrame({"jaccard": jac}).to_csv(out_dir / "jaccard_scores.csv", index=False)
print(results.to_string(index=False))
print(f"\nJaccard: mean={jac.mean():.4f} median={np.median(jac):.4f} max={jac.max():.4f}")

# Reuse the original plotting functions
sys.path.insert(0, script_dir)
try:
    from step10_permutation_test_multiplex import (
        plot_null_distributions, plot_jaccard_distribution)
    plot_null_distributions(true_metrics, null_df.to_dict("records"),
                            pvalues, zscores, str(out_dir))
    plot_jaccard_distribution(jac, str(out_dir))
except Exception as e:
    print(f"Plotting skipped: {e}")


  task_0: 100 perms
  task_1: 100 perms
  task_2: 100 perms
  task_3: 100 perms
  task_4: 100 perms
  task_5: 100 perms
  task_6: 100 perms
  task_7: 100 perms
  task_8: 100 perms
  task_9: 100 perms

Merged 10 tasks -> 1000 permutations
           metric  true_value   null_mean    null_std    z_score  p_value
             size 9940.000000 7602.890000 1146.733721   2.038058    0.981
     edge_density    0.000834    0.002080    0.000076 -16.448660    1.000
mean_internal_deg    8.291147   15.763170    2.121265  -3.522438    1.000
      conductance    0.561782    0.665631    0.047432  -2.189423    0.014
   mean_rwr_score    0.000192    0.000123    0.000017   4.121773    0.000

Jaccard: mean=0.1084 median=0.1087 max=0.1299
Saved null distribution plot → 06_disease_module/permutation_test_merged/permutation_test_null_distributions.png
Saved Jaccard distribution plot → 06_disease_module/permutation_test_merged/permutation_test_jaccard.png
